# Customer Lifetime Value (CLV) - Proxy

This section estimates customer value using a practical proxy approach:
- Historical gross revenue per customer
- Expected value using average monthly retention rate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/clean_retail.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

df.head()

In [ ]:
orders = (
    df.groupby("Invoice", as_index=False)
      .agg(
          OrderValue=("Revenue", "sum"),
          OrderDate=("InvoiceDate", "min"),
          CustomerID=("Customer ID", "first")
      )
)

orders.head()

In [ ]:
customer = (
    orders.groupby("CustomerID", as_index=False)
          .agg(
              HistoricalRevenue=("OrderValue", "sum"),
              Orders=("Invoice", "nunique"),
              FirstOrder=("OrderDate", "min"),
              LastOrder=("OrderDate", "max")
          )
)

customer["AOV"] = customer["HistoricalRevenue"] / customer["Orders"]

customer.head()

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(np.log1p(customer["HistoricalRevenue"]), bins=50)

plt.title("Customer Lifetime Value Distribution (log scale)")
plt.xlabel("log(1 + CLV)")
plt.ylabel("Customers")

plt.show()

In [ ]:
customer_sorted = customer.sort_values("HistoricalRevenue", ascending=False)

customer_sorted["CumRevenueShare"] = (
    customer_sorted["HistoricalRevenue"].cumsum()
    / customer_sorted["HistoricalRevenue"].sum()
)

top_10 = int(len(customer_sorted) * 0.10)

customer_sorted.iloc[top_10]["CumRevenueShare"]